DH演習2026 · 教材 第54版 · 2026-r54 · 2026-11-26

<details><summary>教材の版と相談用情報</summary>

相談時は次の情報を伝えてください。

```text
DH演習2026 · 教材 第54版
教材ID: 2026-r54
教材日付: 2026-11-26
活動: place_identity
環境: Colab / Jupyter
ソース: 9dbb0fdc0476256d55eb9592df1f749ab94cdee9
```

</details>

[同じ版の教材・変更点](https://nlp.lang.osaka-u.ac.jp/dh/practicum/editions.html)


# 第8回：同じ地名は、同じ場所を指すか

確認問題まで上から順にセルを実行してください（Shift+Enter）。1・2節の画像を取得できないときも、3節から続けられます。値を変えたら、そのセルから下を再実行します。判断表の記入と保存は、授業の教材（marimo）で行います。

確認問題の後の追加課題は授業外・任意です。予測とコードの変更は自分で行い、エラーが出たらAIに説明を求められます。

資料：[新板大坂之図](https://image.oml.city.osaka.lg.jp/da/detail?tilcod=0000000021-OSK0186760)と[難波橋の絵葉書](https://image.oml.city.osaka.lg.jp/da/detail?tilcod=0000000021-OSK0157268)（大阪市立図書館、CC0）、[Wikidata](https://www.wikidata.org/)（CC0）

## 1. 古地図で橋名を探す

N1は1657年の原図を1980年に複製した地図で、東が上（北が左）です。次のセルは原画像（約25 MB）を一度だけ取得し、川の部分と橋名の部分を拡大して表示します。提供元の縮小画像では橋名を読めないため、原画像を使います。

In [ ]:
#@title 操作：古地図を取得して拡大する
from pathlib import Path

import requests
from IPython.display import display
from PIL import Image

url = "https://image.oml.city.osaka.lg.jp/da/download/?id=0000000021-OSK0186760&size=org&type=image&file=%2F%E5%86%99%E7%9C%9F%E3%83%BB%E7%B5%B5%E3%81%AF%E3%81%8C%E3%81%8D%E3%83%BB%E5%8F%A4%E6%96%87%E6%9B%B8%E3%83%BB%E5%9C%B0%E5%9B%B3%2Fr1002001.jpg"
n1_path = Path("N1.jpg")
if not n1_path.exists():
    try:
        response = requests.get(url, timeout=180)
        response.raise_for_status()
        n1_path.write_bytes(response.content)
    except requests.RequestException as error:
        print("古地図を取得できませんでした。時間をおいて、このセルを再実行してください。")
        print("提供元の資料ページでも画像を見られます：https://image.oml.city.osaka.lg.jp/da/detail?tilcod=0000000021-OSK0186760")
        print("川と橋名の拡大は、授業の教材（marimo）①にあります。3節から続けられます。")
        print("詳細：", error)
if n1_path.exists():
    try:
        n1 = Image.open(n1_path)
        display(n1.crop((100, 2300, 2000, 3600)).reduce(2), n1.crop((950, 2670, 1250, 2920)))
    except OSError as error:
        n1_path.unlink()
        print("取得したファイルを画像として開けませんでした。このセルを再実行してください。")
        print("提供元の資料ページ：https://image.oml.city.osaka.lg.jp/da/detail?tilcod=0000000021-OSK0186760")
        print("詳細：", error)

## 2. 絵葉書の年と景観を確かめる

N2の目録の刊年は[1915]です。角括弧は目録が補った年を示します。ライオン像・路面電車・欄干を探してください。

In [ ]:
#@title 操作：絵葉書を取得する
url = "https://image.oml.city.osaka.lg.jp/da/download/?id=0000000021-OSK0157268&size=view&type=image&file=%2F%E5%86%99%E7%9C%9F%E3%83%BB%E7%B5%B5%E3%81%AF%E3%81%8C%E3%81%8D%E3%83%BB%E5%8F%A4%E6%96%87%E6%9B%B8%E3%83%BB%E5%9C%B0%E5%9B%B3%2Fc0150001.jpg"
try:
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    Path("N2.jpg").write_bytes(response.content)
except requests.RequestException as error:
    print("絵葉書を取得できませんでした。時間をおいて、このセルを再実行してください。")
    print("提供元の資料ページでも画像を見られます：https://image.oml.city.osaka.lg.jp/da/detail?tilcod=0000000021-OSK0157268")
    print("授業の教材（marimo）①にも同じ絵葉書があります。3節から続けられます。")
    print("詳細：", error)
if Path("N2.jpg").exists():
    try:
        display(Image.open("N2.jpg"))
    except OSError as error:
        Path("N2.jpg").unlink()
        print("取得したファイルを画像として開けませんでした。提供元の資料ページ：https://image.oml.city.osaka.lg.jp/da/detail?tilcod=0000000021-OSK0157268")
        print("詳細：", error)

## 3. Wikidataで候補を検索する

次の取得セルは、`live`が`False`の間、2026年11月25日にWikidataで「難波橋」と「天神橋」を検索した上位5件を表示します。授業の教材（marimo）の同梱の検索結果と同じ候補です。`live`を`True`にすると、検索語をWikidataへ送り、今の結果を取得します。

表には、候補の名前・説明・検索語に一致した表記と、最初の座標を表示します。座標は、現在の橋などを代表する一点です。

In [ ]:
query = "難波橋"
live = False

In [ ]:
#@title 操作：候補の表を用意する（このセルは読まなくてかまいません）
import pandas as pd
import requests

API = "https://www.wikidata.org/w/api.php"
HEADERS = {"User-Agent": "DHEnshu2026Colab/1.0 (https://nlp.lang.osaka-u.ac.jp/dh/practicum/)"}
COLUMNS = ["候補ID", "名前", "説明", "一致した表記", "緯度", "経度"]

# 2026-11-25にWikidataで検索した上位5件（marimo版の同梱の検索結果と同じ）
saved_searches: dict = {
    "難波橋": [
        {"候補ID": "Q11659546", "名前": "難波橋", "説明": "大阪市の橋", "一致した表記": "難波橋", "緯度": 34.692694, "経度": 135.507139},
        {"候補ID": "Q11614009", "名前": "船場", "説明": "大阪市の中心業務地区", "一致した表記": "難波橋筋", "緯度": 34.68388337114979, "経度": 135.50346952385993},
        {"候補ID": "Q858429", "名前": "なにわ橋駅", "説明": "大阪府大阪市北区にある京阪電気鉄道の駅", "一致した表記": "難波橋站", "緯度": 34.692972, "経度": 135.505937},
    ],
    "天神橋": [
        {"候補ID": "Q101792030", "名前": "天神橋", "説明": "日本の大阪府大阪市北区の町", "一致した表記": "天神橋", "緯度": None, "経度": None},
        {"候補ID": "Q11443265", "名前": "天神橋", "説明": "大阪市の橋", "一致した表記": "天神橋", "緯度": 34.69136111, "経度": 135.51172222},
        {"候補ID": "Q141950606", "名前": "天神橋", "説明": "", "一致した表記": "天神橋", "緯度": None, "経度": None},
        {"候補ID": "Q17218475", "名前": "天神橋", "説明": "福島市にある橋", "一致した表記": "天神橋", "緯度": 37.74758333333333, "経度": 140.46755555555555},
        {"候補ID": "Q141944289", "名前": "天神橋", "説明": "", "一致した表記": "天神橋", "緯度": None, "経度": None},
    ],
}


def ask_wikidata(params):
    response = requests.get(API, headers=HEADERS, params={**params, "format": "json"}, timeout=30)
    response.raise_for_status()
    data = response.json()
    if "error" in data:
        raise ValueError(data["error"].get("info", "Wikidataがエラーを返しました"))
    return data


def first_point(entity):
    claims = entity.get("claims") or {}
    points = claims.get("P625") or []
    if not points:
        return None, None
    value = points[0].get("mainsnak", {}).get("datavalue", {}).get("value", {})
    return value.get("latitude"), value.get("longitude")


def search_live(text):
    params = {"action": "wbsearchentities", "search": text, "language": "ja", "uselang": "ja", "type": "item", "limit": 5}
    found = ask_wikidata(params)["search"]
    if not found:
        return []
    ids = "|".join(item["id"] for item in found)
    entities = ask_wikidata({"action": "wbgetentities", "ids": ids, "props": "claims"})["entities"]
    rows = []
    for item in found:
        latitude, longitude = first_point(entities.get(item["id"], {}))
        rows.append({"候補ID": item["id"], "名前": item.get("label", ""), "説明": item.get("description", ""),
                     "一致した表記": item.get("match", {}).get("text", ""), "緯度": latitude, "経度": longitude})
    return rows


search_text = str(query).strip()
candidate_rows = []
if live:
    try:
        candidate_rows = search_live(search_text)
        print("Wikidataで「" + search_text + "」を検索しました。候補：", len(candidate_rows), "件")
    except (requests.RequestException, ValueError, KeyError) as error:
        print("Wikidataから候補を取得できませんでした。liveをFalseに戻すと、保存した結果で続けられます。")
        print("詳細：", error)
elif search_text in saved_searches:
    candidate_rows = saved_searches[search_text]
    print("保存した「" + search_text + "」の検索結果（2026年11月25日取得）を表示します。")
else:
    print("保存した検索結果は「難波橋」と「天神橋」だけです。ほかの地名は、liveをTrueにして検索してください。")
candidates = pd.DataFrame(candidate_rows, columns=COLUMNS)
candidates

## 4. 候補を絞る条件を変える

この節は「難波橋」の候補で試します。`exclude_stations`を`True`にすると、名前に「駅」を含む候補を除きます。`require_bridge`を`True`にすると、説明に「橋」を含まない候補を除きます。

`candidates.to_dict("records")`は、候補の表の各行を、列名をキーとする辞書にしたリストです。`and`は二つの条件がともに成り立つときだけ真になり、`not in`は指定した文字を含まないかを調べます。`if`の条件が成り立つと`elif`は調べません。`kept`には名前と候補IDを並べて残すので、同じ名前の候補も区別できます。

まず`exclude_stations`だけを`True`にして実行し、外れた候補と理由を確認してください。なにわ橋駅は、名前に「駅」があり、説明には「橋」がありません。`require_bridge`も`True`にして実行する前に、なにわ橋駅がどちらの理由で除かれるかを予測してください。説明が空だったり「橋」の字がなかったりすると、`require_bridge`は橋の項目でも除きます。

In [ ]:
exclude_stations = False
require_bridge = False

kept = []
for candidate in candidates.to_dict("records"):
    name = candidate["名前"] + "（" + candidate["候補ID"] + "）"
    if exclude_stations and "駅" in candidate["名前"]:
        print("除く：", name, "名前に「駅」")
    elif require_bridge and "橋" not in candidate["説明"]:
        print("除く：", name, "説明に「橋」がない")
    else:
        kept.append(name)
print("全候補：", len(candidates), "残った候補：", kept)

## 5. 判断の基準を確かめる

同じ読みでも別の対象を指し、違う名前でも同じ対象を指すことがあります。中之島公園の案内図の写真は、授業の教材（marimo）①にあります。

| 表記 | 指しているもの | 表に残す関係 |
| --- | --- | --- |
| 難波橋 | 川を渡る橋 | 橋の名前 |
| ライオン橋 | 同じ橋の愛称 | 橋の別名 |
| なにわ橋駅 | 京阪中之島線の駅 | 橋とは別の対象 |

残った候補が資料の場所を指すかは、marimo版③の判断表に、判断した資料と理由とともに記します。

## 確認問題

3節の`query`を「天神橋」に変えて、3節から実行し直してください。N3の[大阪市の市街地図](https://www.city.osaka.lg.jp/kensetsu/cmsfiles/contents/0000032/32065/shigaizu_1.jpg)では、難波橋の東に天神橋が描かれています。説明を読み、（1）N3の橋に当たる候補と、（2）説明だけではN3の橋かどうか判断できない候補を、一つずつ答えてください。説明からN3の橋と別だと分かる候補は、どちらにも入りません。

- （1）`bridge_id`：N3の天神橋に当たる候補のID
- （2）`open_id`：説明だけではN3の橋かどうか判断できない候補のID（一つ）
- `clue`：判断した手がかり（任意）

In [ ]:
bridge_id = ""
open_id = ""
clue = ""

In [ ]:
#@title 答えを確かめる
import re
import unicodedata

bridge_hints = {
    "Q11659546": "難波橋の候補IDです。N3で難波橋の東に描かれた天神橋の候補を選んでください",
    "Q17218475": "説明は「福島市にある橋」です。N3は大阪市の地図です",
    "Q101792030": "説明は「日本の大阪府大阪市北区の町」です。N3で選ぶのは、川を渡る橋です",
    "Q141950606": "説明が空欄の候補です。説明から大阪市の橋と確かめられる候補を選んでください",
    "Q141944289": "説明が空欄の候補です。説明から大阪市の橋と確かめられる候補を選んでください",
}
open_hints = {
    "Q11443265": "説明が「大阪市の橋」なので、N3の橋に当たると判断できます。（2）では、説明だけでは判断できない候補を選んでください",
    "Q17218475": "説明が「福島市にある橋」なので、N3の橋とは別だと判断できます。説明だけでは判断できない候補を選んでください",
    "Q101792030": "説明が「日本の大阪府大阪市北区の町」なので、川を渡るN3の橋とは別だと判断できます。説明だけでは判断できない候補を選んでください",
    "Q11659546": "難波橋の候補IDです。「天神橋」の表から、説明だけでは判断できない候補を選んでください",
}
bridge_value = globals().get("bridge_id")
open_value = globals().get("open_id")
result = "結果：合格"
if not (isinstance(bridge_value, str) and isinstance(open_value, str)):
    result = '結果：不合格（`bridge_id = ""`と`open_id = ""`の形のまま、""の中に候補IDを入れてください）'
elif not (bridge_value.strip() or open_value.strip()):
    result = "未回答：答えを記入してください。"
else:
    checks = [
        ("（1）", bridge_value, bridge_hints, ["Q11443265"]),
        ("（2）", open_value, open_hints, ["Q141950606", "Q141944289"]),
    ]
    for label, value, hints, expected in checks:
        qid = unicodedata.normalize("NFKC", value).strip().strip("「」`'\" ").upper()
        if not re.fullmatch(r"Q[1-9][0-9]*", qid):
            result = "結果：不合格（" + label + "の候補IDは「Q」で始まる番号です。表の「候補ID」の列から写してください）"
        elif qid in hints:
            result = "結果：不合格（" + label + "：" + hints[qid] + "）"
        elif qid not in expected:
            result = "結果：不合格（" + label + "：「天神橋」の検索結果にない候補IDです。表の「候補ID」の列から写してください）"
        else:
            continue
        break
print(result)
if result == "結果：合格":
    print()
    print("考え方：5件の「一致した表記」は、どれも「天神橋」なので、説明を読みます。")
    print("（1）説明が「大阪市の橋」のQ11443265が、N3に描かれた橋に当たります。")
    print("Q17218475（福島市にある橋）とQ101792030（大阪市北区の町）は、説明がN3の橋と合わないので、対応を却下できます。")
    print("（2）説明が空欄のQ141950606とQ141944289は、この表だけでは判断できません。分類や所在する行政区を出典で確かめます。")
    if str(globals().get("clue") or "").strip():
        print("clueに書いた手がかりを、この考え方と比べてください。")

## 授業外・任意：名前の代わりに分類で候補を選ぶ

Wikidataは、項目が何であるかを`wdt:P31`（分類）で、分類の上下関係を`wdt:P279`（上位の分類）で記録します。分類とその上下関係を定めた語彙をオントロジーと呼びます。下のセルでは、3節の候補の分類を表にし、`kind`（`Q12280`は橋）に当たる候補を選びます。

既定の表は、3節の保存した検索結果（「難波橋」「天神橋」）の候補について、Wikidataから取得した分類です（難波橋は2026年11月25日、天神橋は2026年11月26日取得）。分類が記録されていない候補は、表に出ません。現在のデータで試すときは、3節の`live`を`True`にして、3節から再実行します。

まず表を表示し、下の選択セルを実行してください。「上位の分類」は各候補の分類から一段たどった先で、`follow_subclasses`を`True`にすると上位の分類が`kind`の候補も選ばれます。表の「上位の分類のID」から新たに加わる候補を予測してから`True`に変え、名前に「橋」を含むのに分類では選ばれない候補を一つ挙げてください。一つの候補に分類や上位の分類が複数あると表に複数の行が出るため、`name not in selected`で同じ候補を二度加えないようにします。

`name`は名前と候補IDを並べた文字列なので、名前がどれも「天神橋」の候補も区別できます。

分類はWikidataの編集者の判断です。資料の場所との同定は、授業の教材（marimo）③の判断表に、判断した資料と理由を書いて行います。

In [ ]:
#@title 操作：分類の表を用意する
import requests

columns = ["候補ID", "名前", "分類のID", "分類", "上位の分類のID", "上位の分類"]
# Wikidataから取得した分類（難波橋は2026-11-25、天神橋は2026-11-26。3節の保存した検索結果と同じ候補）
saved_classes = {
    "難波橋": [
        ("Q11659546", "難波橋", "Q158438", "アーチ橋", "Q12280", "橋"),
        ("Q11659546", "難波橋", "Q537127", "道路橋", "Q12280", "橋"),
        ("Q11614009", "船場", "Q738570", "中心業務地区", "Q123705", "都市の地区"),
        ("Q11614009", "船場", "Q738570", "中心業務地区", "Q1133961", "商業地域"),
        ("Q11614009", "船場", "Q738570", "中心業務地区", "Q28692737", "オフィス街"),
        ("Q11614009", "船場", "Q738570", "中心業務地区", "Q106860125", "center"),
        ("Q858429", "なにわ橋駅", "Q55488", "鉄道駅", "Q548662", "停車場"),
        ("Q858429", "なにわ橋駅", "Q55488", "鉄道駅", "Q800279", "鉄道施設"),
        ("Q858429", "なにわ橋駅", "Q55488", "鉄道駅", "Q97987466", "operating control point"),
        ("Q858429", "なにわ橋駅", "Q55488", "鉄道駅", "Q124673697", "rail transport station"),
        ("Q858429", "なにわ橋駅", "Q22808403", "地下駅", "Q863404", "地下"),
        ("Q858429", "なにわ橋駅", "Q22808403", "地下駅", "Q12819564", "駅"),
    ],
    "天神橋": [
        ("Q101792030", "天神橋", "Q5327369", "町丁", "Q486972", "集落"),
        ("Q101792030", "天神橋", "Q5327369", "町丁", "Q65948724", "町・字"),
        ("Q11443265", "天神橋", "Q12280", "橋", "Q83620", "道"),
        ("Q11443265", "天神橋", "Q12280", "橋", "Q618123", "地理的特徴"),
        ("Q11443265", "天神橋", "Q12280", "橋", "Q811979", "構造物"),
        ("Q11443265", "天神橋", "Q158438", "アーチ橋", "Q12280", "橋"),
        ("Q17218475", "天神橋", "Q12280", "橋", "Q83620", "道"),
        ("Q17218475", "天神橋", "Q12280", "橋", "Q618123", "地理的特徴"),
        ("Q17218475", "天神橋", "Q12280", "橋", "Q811979", "構造物"),
    ],
}
class_rows = []
if candidates.empty:
    print("3節の候補の表が空です。3節の`query`と`live`を確かめて、3節から実行し直してください。")
elif live:
    names = candidates.drop_duplicates("候補ID").set_index("候補ID")["名前"].to_dict()
    sparql = "SELECT ?item ?class ?classLabel ?parent ?parentLabel WHERE { VALUES ?item { " + " ".join("wd:" + i for i in names) + " } ?item wdt:P31 ?class . OPTIONAL { ?class wdt:P279 ?parent . } SERVICE wikibase:label { bd:serviceParam wikibase:language 'ja,en'. } }"
    try:
        response = requests.get("https://query.wikidata.org/sparql", params={"query": sparql, "format": "json"}, headers=HEADERS, timeout=60)
        response.raise_for_status()
        bindings = response.json()["results"]["bindings"]
    except (requests.RequestException, ValueError, KeyError) as error:
        bindings = []
        print("Wikidataから分類を取得できませんでした。3節のliveをFalseに戻すと、保存した分類で続けられます。")
        print("詳細：", error)
    for row in bindings:
        fields = {key: row.get(key, {}).get("value", "") for key in ("item", "class", "classLabel", "parent", "parentLabel")}
        qid = fields["item"].rsplit("/", 1)[-1]
        class_rows.append({
            "候補ID": qid,
            "名前": names.get(qid, qid),
            "分類のID": fields["class"].rsplit("/", 1)[-1],
            "分類": fields["classLabel"],
            "上位の分類のID": fields["parent"].rsplit("/", 1)[-1],
            "上位の分類": fields["parentLabel"],
        })
elif search_text in saved_classes:
    class_rows = [dict(zip(columns, values, strict=True)) for values in saved_classes[search_text]]
else:
    print("保存した分類は「難波橋」と「天神橋」の候補だけです。ほかの地名は、3節のliveをTrueにして、3節から実行し直してください。")
pd.DataFrame(class_rows, columns=columns)

In [ ]:
kind = "Q12280"  # 橋
follow_subclasses = False

named = []
selected = []
for row in class_rows:
    name = row["名前"] + "（" + row["候補ID"] + "）"
    if "橋" in row["名前"] and name not in named:
        named.append(name)
    is_direct = row["分類のID"] == kind
    is_parent = follow_subclasses and row["上位の分類のID"] == kind
    if (is_direct or is_parent) and name not in selected:
        selected.append(name)
print("名前に「橋」を含む候補：", named)
print("分類で選んだ候補：", selected)